In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.roi_heads import RoIHeads
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

In [ ]:
# 1. Define the Custom Part-Aware Head
class PartAwareRoIHeads(RoIHeads):
    def __init__(self, *args, num_parts=0, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_parts = num_parts
        
        # This is the new branch: It takes the same features as the box classifier
        # The representation size usually matches the box_head output (e.g., 1024 for ResNet50)
        self.part_predictor = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_parts)  # Output logic for each part (Multi-label)
        )

    def forward(self, features, proposals, image_shapes, targets=None):
        """
        Overriding the forward pass to inject Part Loss.
        """
        if targets is not None:
            for t in targets:
                # IMPORTANT: Ensure your dataset loader adds this 'part_labels' key!
                # Format: Tensor of shape [N_boxes, num_parts] (0 or 1 for presence)
                assert "part_labels" in t, "Targets must contain 'part_labels' for training"

        # --- Standard Faster R-CNN Logic (Copied logic to access features) ---
        
        # 1. RoI Pooling
        box_features = self.box_roi_pool(features, proposals, image_shapes)
        
        # 2. Feature Extraction (Two MLP layers usually)
        box_features = self.box_head(box_features)
        
        # 3. Standard Predictions (Class + BBox)
        class_logits, box_regression = self.box_predictor(box_features)

        # --- NEW: Part Prediction Logic ---
        part_logits = self.part_predictor(box_features)

        result = torch.jit.annotate(list[dict[str, torch.Tensor]], [])
        losses = {}

        if self.training:
            assert targets is not None
            # Match proposals to ground truth to get the targets for the loss
            matched_idxs, labels = self.assign_targets_to_proposals(proposals, targets)
            
            # Prepare standard detection losses
            regression_targets = self.box_coder.encode(matched_idxs, proposals, targets)
            loss_classifier, loss_box_reg = self.box_loss(
                class_logits, box_regression, labels, regression_targets
            )
            
            # --- NEW: Calculate Part Loss ---
            # We need to fetch the Ground Truth part labels for the sampled proposals
            part_loss = self.compute_part_loss(part_logits, matched_idxs, targets, labels)

            losses = {
                "loss_classifier": loss_classifier,
                "loss_box_reg": loss_box_reg,
                "loss_part": part_loss * 0.5  # Weighting factor (Tune this!)
            }
        else:
            boxes, scores, labels = self.postprocess_detections(class_logits, box_regression, proposals, image_shapes)
            num_images = len(boxes)
            for i in range(num_images):
                result.append(
                    {
                        "boxes": boxes[i],
                        "labels": labels[i],
                        "scores": scores[i],
                        # Optional: Return part probabilities at inference
                        "part_probs": torch.sigmoid(part_logits).split([len(b) for b in proposals])[i]
                    }
                )

        return result, losses

    def compute_part_loss(self, part_logits, matched_idxs, targets, gt_labels):
        """
        Computes Binary Cross Entropy for part presence.
        Only computes loss for Positive Samples (Foreground objects).
        """
        part_targets_list = []
        
        for i, (idxs, t) in enumerate(zip(matched_idxs, targets)):
            # Extract the part labels for the matched GT objects
            # t['part_labels'] should be [Num_GT_Boxes, Num_Parts]
            gt_parts = t['part_labels'][idxs]
            part_targets_list.append(gt_parts)
            
        part_targets = torch.cat(part_targets_list, dim=0)

        # Filter: We only care about part predictions for Foreground objects (label > 0)
        # gt_labels includes background (0). 
        foreground_mask = gt_labels > 0
        
        if foreground_mask.sum() == 0:
            return part_logits.sum() * 0 # No loss if no foreground

        # Slice logits and targets to only include foreground
        relevant_logits = part_logits[foreground_mask]
        relevant_targets = part_targets[foreground_mask].float()

        # BCE Loss: Predict presence/absence of parts
        return F.binary_cross_entropy_with_logits(relevant_logits, relevant_targets)

In [ ]:
# 2. Helper to Initialize the Model
def get_part_aware_model(num_classes, num_parts):
    # Load backbone
    backbone = resnet_fpn_backbone('resnet50', pretrained=True)
    
    # Create the model using standard FasterRCNN but inject our custom RoIHeads
    model = FasterRCNN(backbone, num_classes=num_classes)
    
    # Replace the standard RoIHeads with our Custom one
    # We must preserve the internal sub-modules (box_roi_pool, box_head, etc.)
    model.roi_heads = PartAwareRoIHeads(
        # Pass existing components from the initialized standard model
        box_roi_pool=model.roi_heads.box_roi_pool,
        box_head=model.roi_heads.box_head,
        box_predictor=model.roi_heads.box_predictor,
        # Standard args
        fg_iou_thresh=0.5, bg_iou_thresh=0.5,
        batch_size_per_image=512, positive_fraction=0.25,
        bbox_reg_weights=None,
        score_thresh=0.05, nms_thresh=0.5, detections_per_img=100,
        # Custom args
        num_parts=num_parts
    )
    
    return model

# --- Example Usage ---
# CUB has 11 parts -> num_parts=11
# Fish-Vista has 9 parts -> num_parts=9
model = get_part_aware_model(num_classes=200, num_parts=11)
# print(model) # You will see the new 'part_predictor' layer